In [ ]:
from ngsolve import *
from netgen.geom2d import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np

# ------------------------------------------------------------
# Parameter
# ------------------------------------------------------------
# H, L = 4.0, 28.0
# cx, cy, R = 7.0, 0.0, 0.5
H, L = 0.41, 2.0
cx, cy, R = 0.5, 0.2, 0.05

nu = 1e-3       #viscosity

# ------------------------------------------------------------
# Geometrie
# Rectangle(width, height) liegt standardmäßig in [0,L]×[0,H]
# → daher zuerst erzeugen, dann verschieben
# ------------------------------------------------------------
rect = MoveTo(0,0).Rectangle(L, H).Face()

rect.edges.Min(X).name = "inlet"
rect.edges.Max(X).name = "outlet"
rect.edges.Min(Y).name = "walls"
rect.edges.Max(Y).name = "walls"

cyl = Circle((cx,cy), R).Face()
cyl.edges.name = "obstacle"

shape = rect - cyl
#Draw(shape)

# ------------------------------------------------------------
# Mesh
# ------------------------------------------------------------
geo = OCCGeometry(shape,dim=2)
mesh = Mesh(geo.GenerateMesh(maxh=0.05))  

mesh.Refine()
Draw(mesh);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [ ]:
# ------------------------------------------------------------
# FE-Räume (Taylor–Hood)
# ------------------------------------------------------------
V = VectorH1(mesh, order=2, dirichlet="inlet|walls|obstacle")
Q = H1(mesh, order=1, dirichlet="outlet")   

X = FESpace([V, Q])
(u, p) = X.TrialFunction()
(v, q) = X.TestFunction()

gfu = GridFunction(X)
gfu_u, gfu_p = gfu.components


In [ ]:
# ------------------------------------------------------------
# Inlet-Profil (parabolisch auf [-H/2, H/2])            -> Randbedingungen bei Inlet:  u = uin bei "inlet"
# ------------------------------------------------------------
Umax = 2.0             

# Parabolisches Profil
uin_x = (Umax*4*y*(H-y)/(H*H))

# Kleine Störung hinzufügen um Wirbelablösung zu initiieren
uin_y = 0.01*sin(20*x)*exp(-10*(y-H/2)**2)  # Kleine vertikale Störung

uin = CoefficientFunction((uin_x, uin_y))

gfu_u.Set(uin, definedon=mesh.Boundaries("inlet"))


In [ ]:
# Reynolds Zahl ausgeben
Reynold = Umax * 2*R/nu
print("Reynolds-Zahl = ",Reynold)

Reynolds-Zahl =  100.0


In [ ]:
# ------------------------------------------------------------
# Stokes: schwache Form
# ------------------------------------------------------------

#für stabilität
alpha = 1e-10
gamma = 0


a = BilinearForm(X)
a += nu*InnerProduct(Grad(u), Grad(v)) * dx         
a += gamma * div(u) * div(v) * dx
a += -div(v)*p * dx
a += -div(u)*q * dx
a += alpha * p*q * dx

L = LinearForm(X)   # keine Volumenkräfte

a.Assemble()
L.Assemble()

# ------------------------------------------------------------
# Lösen -- hatte starke Probleme beim lösen!!!
# ------------------------------------------------------------

inv_stokes = a.mat.Inverse(X.FreeDofs())                

res = L.vec - a.mat*gfu.vec
gfu.vec.data += inv_stokes * res

#Draw (gfu.components[0], mesh);
# ------------------------------------------------------------
# Visualisierung
# ------------------------------------------------------------

Draw(gfu_u, mesh, "velocity")
Draw(gfu_p, mesh, "pressure")

# Stokes hier fertig gelöst -> jetzt gehts an NavierStokes


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [ ]:
# def ApplyDirichlet(gf_u, gf_p):
    # Geschwindigkeit: inlet vorgeben, Wände+Zylinder no-slip
    # gf_u.Set(uin, definedon=mesh.Boundaries("inlet"))
    #gf_u.Set(CoefficientFunction((0,0)), definedon=mesh.Boundaries("walls|obstacle"))
    # Druckreferenz: p=0 am outlet
    #gf_p.Set(0, definedon=mesh.Boundaries("outlet"))

# -----------------------------
# 2) Navier–Stokes via Picard(Oseen) pro Zeitschritt
#     Fixpunkt: w^k = u^k
# -----------------------------

print("Reynolds-Zahl=",Reynold)



t = 0.0
i = 0
dt   = 0.005  # Kleinerer Zeitschritt für bessere Zeitauflösung
tend = 7.0   # Längere Simulation um Wirbelstraße zu entwickeln
picard_maxit = 1
picard_tol   = 1e-6

#für endsimulation
gfut = GridFunction(V, multidim=0)
vel = gfu.components[0]

scene = Draw(gfu_u,mesh,min=0,max=3)

while t < tend - 1e-12:
    # u^n
    u_prev = GridFunction(V)
    u_prev.vec.data = gfu_u.vec

    # Start für Picard: u^{n+1,0} := u^n
    gfu_new = GridFunction(X)
    gfu_new.components[0].vec.data = u_prev.vec
    gfu_new.components[1].vec.data = gfu_p.vec

    # Konvektionsfeld w^0
    w = GridFunction(V)
    w.vec.data = u_prev.vec

    for k in range(picard_maxit):
        w_old = GridFunction(V)
        w_old.vec.data = w.vec

        a = BilinearForm(X, symmetric=False)
        # Zeit: (1/dt)(u^{n+1},v)
        a += (1/dt)*InnerProduct(u, v)*dx
        # Viskosität
        a += nu*InnerProduct(Grad(u), Grad(v))*dx
        # Oseen-Konvektion: ((w·∇)u, v) = (Grad(u)*w, v)
        a += InnerProduct(Grad(u)*w, v)*dx
        # Inkompressibilität (Vorzeichen wie Stokes!)
        a += -div(v)*p*dx
        a += -div(u)*q*dx
        a += alpha * p*q * dx

        f = LinearForm(X)
        # RHS: (1/dt)(u^n, v)
        f += (1/dt)*InnerProduct(u_prev, v)*dx

        a.Assemble()
        f.Assemble()

        # Dirichlet-Werte vor dem Solve setzen
        gfu_new.components[0].Set(uin, definedon=mesh.Boundaries("inlet"))

        # Lösen mit Elimination: erst Residuum, dann Korrektur auf FreeDofs
        inv = a.mat.Inverse(X.FreeDofs())
        res = f.vec - a.mat * gfu_new.vec
        gfu_new.vec.data += inv * res

        # Fixpunkt-Update
        w.vec.data = gfu_new.components[0].vec

        # Konvergenztest
        diff = Norm(w.vec - w_old.vec)       
        if diff < picard_tol:
            break

    # Zeitschritt übernehmen
    gfu.vec.data = gfu_new.vec
    t += dt
    i += 1

    if i%5 == 0: scene.Redraw()
    if i%20 == 0: 
        gfut.AddMultiDimComponent(vel.vec)
        print(f"t = {t:.3f}, Picard iterations: {k+1}", end='\r')

    # scene.Redraw()

# Ergebnis
Draw(gfu.components[0], mesh, "u_ns")
# Draw(gfu.components[1], mesh, "p_ns")


Reynolds-Zahl= 100.0


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [ ]:
Draw (gfut, mesh, interpolate_multidim=True, animate=True);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [ ]:
# Wirbelstärke (Vorticity) visualisieren - zeigt die Wirbelstraße deutlich
from ngsolve import *

# Berechne curl (Rotation) des Geschwindigkeitsfelds
# In 2D: curl(u) = du_y/dx - du_x/dy
u_ns = gfu.components[0]
curl_u = u_ns[1].Diff(x) - u_ns[0].Diff(y)

Draw(curl_u, mesh, "vorticity", min=-50, max=50, deformation=True)
print("Die Wirbelstärke (Vorticity) zeigt die Kármánsche Wirbelstraße!")

Die Wirbelstärke (Vorticity) zeigt die Kármánsche Wirbelstraße!


In [ ]:
# def ApplyDirichlet(gf_u, gf_p):
#     # Geschwindigkeit: inlet vorgeben, Wände+Zylinder no-slip
#     gf_u.Set(uin, definedon=mesh.Boundaries("inlet"))
#     #gf_u.Set(CoefficientFunction((0,0)), definedon=mesh.Boundaries("walls|obstacle"))
#     # Druckreferenz: p=0 am outlet
#     #gf_p.Set(0, definedon=mesh.Boundaries("outlet"))

# # -----------------------------
# # 2) Navier–Stokes via Picard(Oseen) pro Zeitschritt
# #     Fixpunkt: w^k = u^k
# # -----------------------------

# print("Reynolds-Zahl=",Reynold)

# t = 0.0
# i = 0
# dt   = 0.005  # Kleinerer Zeitschritt für bessere Zeitauflösung
# tend = 7.0   # Längere Simulation um Wirbelstraße zu entwickeln
# picard_maxit = 3
# picard_tol   = 1e-6

# underrelaxation_factor = 1      # u_k+1 = u_k + x*A_inv(f-A*u_k)

# #für endsimulation
# gfut = GridFunction(V, multidim=0)
# vel = gfu.components[0]

# scene = Draw(gfu_u,mesh,min=0,max=3)

# while t < tend - 1e-12:

#     # u^n
#     u_prev = GridFunction(V)
#     u_prev.vec.data = gfu_u.vec

#     # Start für Picard: u^{n+1,0} := u^n
#     gfu_new = GridFunction(X)
#     gfu_new.components[0].vec.data = u_prev.vec
#     gfu_new.components[1].vec.data = gfu_p.vec

#     #ApplyDirichlet(gfu_new.components[0], gfu_new.components[1])

#     # Konvektionsfeld w^0
#     w = GridFunction(V)
#     w.vec.data = u_prev.vec

#     for k in range(picard_maxit):
#         w_old = GridFunction(V)
#         w_old.vec.data = w.vec

#         a = BilinearForm(X, symmetric=False)
#         # Zeit: (1/dt)(u^{n+1},v)
#         a += (1/dt)*InnerProduct(u, v)*dx
#         # Viskosität
#         a += nu*InnerProduct(Grad(u), Grad(v))*dx
#         # Oseen-Konvektion: ((w·∇)u, v) = (Grad(u)*w, v)
#         a += InnerProduct(Grad(u)*w, v)*dx
#         # Inkompressibilität (Vorzeichen wie Stokes!)
#         a += -div(v)*p*dx
#         a += -div(u)*q*dx
#         a += alpha * p*q * dx


#         f = LinearForm(X)
#         # RHS: (1/dt)(u^n, v)
#         f += (1/dt)*InnerProduct(u_prev, v)*dx

#         a.Assemble()
#         f.Assemble()

#         # Dirichlet-Werte müssen VOR dem Solve gesetzt sein
#         ApplyDirichlet(gfu_new.components[0], gfu_new.components[1])

#         # Lösen mit Elimination: erst Residuum, dann Korrektur auf FreeDofs
#         inv = a.mat.Inverse(X.FreeDofs())
#         res = f.vec - a.mat * gfu_new.vec
#         gfu_new.vec.data += underrelaxation_factor * inv * res

#         # Fixpunkt-Update
#         w.vec.data = gfu_new.components[0].vec

#         # Konvergenztest
#         diff = Norm(w.vec - w_old.vec)       
#         if diff < picard_tol:
#             break

#     # Zeitschritt übernehmen
#     gfu.vec.data = gfu_new.vec
#     t += dt
#     i += 1

#     if i%5 == 0: scene.Redraw()
#     if i%20 == 0: 
#         gfut.AddMultiDimComponent(vel.vec)
#         print(f"t = {t:.3f}, Picard iterations: {k+1}", end='\r')

#     # scene.Redraw()

# # Ergebnis
# Draw(gfu.components[0], mesh, "u_ns")
# # Draw(gfu.components[1], mesh, "p_ns")
